# <font color="#418FDE" size="6.5" uppercase>**Capstone Case**</font>

>Last update: 20260610.
    
By the end of this Lecture, you will be able to:
- Build a complete Python program that processes a small medical dataset from input to report output. 
- Evaluate the program for correctness, readability, error handling, and appropriate handling of sensitive examples. 
- Present the workflow results in a concise form that supports clinical or research interpretation. 


## **1. Integrated Data Workflow**

### **1.1. Data Input Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_01_01.jpg?v=1781149609" width="250">



>* Load medical data from reliable sources
>* Check files and columns before analysis

>* Check types, missing values, and sensitive fields
>* Flag input problems before analysis begins

>* Document sources, assumptions, and privacy protections
>* Summarize inputs to support trustworthy workflows



In [ ]:
#@title Python Code - Data Input Pipeline

# Build a reproducible medical input pipeline.
# Validate structure before interpreting clinical values.
# Summarize safely without exposing sensitive details.

import csv
import io
from datetime import datetime

# Store tiny synthetic records as comma separated text.
csv_text = """patient_id,age,sex,visit_date,systolic_bp,glucose_mg_dl,diagnosis
P001,67,F,2025-01-10,142,126,hypertension
P002,54,M,2025-01-11,136,not available,diabetes
P003,72,F,2025-01-12,155,164,hypertension
P004,45,X,2025-01-13,128,101,asthma
P005,60,M,2025-01-14,,139,diabetes
"""

# Define fields required by the later workflow.
required_fields = {
    "patient_id", "age", "sex", "visit_date",
    "systolic_bp", "glucose_mg_dl", "diagnosis"
}

# Open the external source through memory safely.
source_file = io.StringIO(csv_text)
reader = csv.DictReader(source_file)

# Check whether expected columns are available.
found_fields = set(reader.fieldnames or [])
missing_fields = sorted(required_fields - found_fields)

# Convert rows while collecting input warnings.
clean_rows = []
warnings = []

# Process each row with careful type checks.
for row_number, row in enumerate(reader, start=2):
    clean = dict(row)
    clean["age"] = int(row["age"])

    # Convert visit date into a date object.
    clean["visit_date"] = datetime.strptime(
        row["visit_date"], "%Y-%m-%d"
    ).date()

    # Convert numeric clinical fields when possible.
    for field in ["systolic_bp", "glucose_mg_dl"]:
        raw_value = row[field].strip()

        if raw_value.isdigit():
            clean[field] = int(raw_value)
        else:
            clean[field] = None
            warnings.append(f"row {row_number}: {field} missing or nonnumeric")

    # Flag unexpected categorical coding early.
    if clean["sex"] not in {"F", "M"}:
        warnings.append(f"row {row_number}: sex code needs review")

    clean_rows.append(clean)

# Create concise, privacy-preserving report values.
record_count = len(clean_rows)
valid_bp_count = sum(row["systolic_bp"] is not None for row in clean_rows)
valid_glucose_count = sum(row["glucose_mg_dl"] is not None for row in clean_rows)

# Count diagnoses without printing patient identifiers.
diagnosis_counts = {}
for row in clean_rows:
    diagnosis = row["diagnosis"]
    diagnosis_counts[diagnosis] = diagnosis_counts.get(diagnosis, 0) + 1

# Print a compact input pipeline report.
print("Data input pipeline report")
print(f"Records loaded: {record_count}")
print(f"Required columns missing: {missing_fields}")
print(f"Valid blood pressure values: {valid_bp_count}")
print(f"Valid glucose values: {valid_glucose_count}")
print(f"Diagnosis counts: {diagnosis_counts}")
print(f"Input warnings found: {len(warnings)}")
print("Review examples:", warnings[:3])



### **1.2. Data Processing Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_01_02.jpg?v=1781149611" width="250">



>* Check fields and formats before summarizing
>* Prevent data errors from misleading interpretation

>* Separate cleaning, validation, transformation, and summarization
>* Organized steps improve testing, clarity, adaptability

>* Protect privacy with aggregate medical summaries
>* Report errors without exposing sensitive records



In [ ]:
#@title Python Code - Data Processing Pipeline

# Build one complete medical data pipeline.
# Keep records fictional and safely summarized.
# Validate, clean, transform, then report.

import pandas as pd
import numpy as np

# Create a tiny fictional encounter dataset.
records = [
    {"id": "A01", "age": 72, "setting": "ED", "diagnosis": "diabetes", "systolic": "142", "a1c": "8.1", "outcome": "follow-up"},
    {"id": "A02", "age": 64, "setting": "outpatient", "diagnosis": "Diabetes ", "systolic": "128", "a1c": "7.2", "outcome": "stable"},

    {"id": "A03", "age": 55, "setting": "inpatient", "diagnosis": "hypertension", "systolic": "169", "a1c": "5.7", "outcome": "improved"},
    {"id": "A03", "age": 55, "setting": "inpatient", "diagnosis": "hypertension", "systolic": "169", "a1c": "5.7", "outcome": "improved"},

    {"id": "A04", "age": -4, "setting": "clinic", "diagnosis": "asthma", "systolic": "118", "a1c": "5.3", "outcome": "stable"},
    {"id": "A05", "age": 81, "setting": "ER", "diagnosis": "cardiac", "systolic": "missing", "a1c": "6.4", "outcome": "monitor"},
]

# Load records into a small dataframe.
df = pd.DataFrame(records)
expected = ["id", "age", "setting", "diagnosis", "systolic", "a1c", "outcome"]
missing_columns = sorted(set(expected) - set(df.columns))

# Stop early if required columns are absent.
if missing_columns:
    raise ValueError("Required fields are missing from dataset.")

# Remove duplicates before grouped summaries.
initial_rows = len(df.index)
df = df.drop_duplicates(subset=["id"]).copy()
duplicate_count = initial_rows - len(df.index)

# Standardize labels without exposing full records.
df["setting"] = df["setting"].str.strip().str.lower()
df["diagnosis"] = df["diagnosis"].str.strip().str.lower()
setting_map = {"ed": "emergency", "er": "emergency", "clinic": "outpatient"}

# Apply setting mapping with safe fallback.
df["setting"] = df["setting"].replace(setting_map)
valid_settings = {"emergency", "outpatient", "inpatient"}
invalid_settings = (~df["setting"].isin(valid_settings)).sum()

# Convert clinical measurements into numeric values.
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["systolic"] = pd.to_numeric(df["systolic"], errors="coerce")
df["a1c"] = pd.to_numeric(df["a1c"], errors="coerce")

# Flag missing or implausible values conservatively.
invalid_age = ((df["age"] < 0) | (df["age"] > 120)).sum()
missing_systolic = df["systolic"].isna().sum()
missing_a1c = df["a1c"].isna().sum()

# Keep valid rows for report-ready summaries.
valid_mask = df["age"].between(0, 120)
valid_mask &= df["setting"].isin(valid_settings)
valid_mask &= df["systolic"].notna()
clean = df.loc[valid_mask].copy()

# Derive an interpretable clinical indicator.
clean["high_systolic"] = clean["systolic"] >= 140
summary = clean.groupby("setting")["high_systolic"].agg(["count", "mean"])
summary["percent_high"] = (summary["mean"] * 100).round(1)

# Print concise aggregate report only.
print("Integrated Data Workflow Report")
print(f"Input rows: {initial_rows}; usable rows: {len(clean.index)}")
print(f"Duplicates removed: {duplicate_count}")
print(f"Invalid ages: {invalid_age}; invalid settings: {invalid_settings}")
print(f"Missing systolic: {missing_systolic}; missing A1c: {missing_a1c}")

# Present grouped results without individual records.
print("High systolic percentage by setting:")
print(summary[["count", "percent_high"]].to_string())



### **1.3. Report Generation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_01_03.jpg?v=1781149613" width="250">



>* Turn processed data into clear reports
>* Summarize key findings for interpretation

>* Include context, exclusions, and limitations
>* Protect privacy with de-identified summaries

>* Make reports concise, consistent, and reproducible
>* Turn processed data into responsible interpretation



In [ ]:
#@title Python Code - Report Generation

# Create a concise clinical report.
# Use aggregated deidentified example data.
# Keep workflow readable and reproducible.

import pandas as pd
import io as text_io

# Store tiny deidentified clinic data.
csv_text = """patient_code,systolic,diastolic,age,visit_status
P001,128,78,54,complete
P002,145,92,67,complete
P003,,85,61,incomplete
P004,181,111,70,complete
P005,118,76,45,complete
P006,132,84,,complete
P007,250,82,50,complete
P008,139,88,58,complete
"""

# Load records from text safely.
records = pd.read_csv(text_io.StringIO(csv_text))

# Count starting records for reporting.
total_records = len(records.index)

# Define plausible adult blood pressure limits.
valid_pressure = (
    records["systolic"].between(70, 220)
    & records["diastolic"].between(40, 130)
)

# Require age for interpretable summary results.
valid_age = records["age"].notna()

# Keep valid rows without exposing identifiers.
valid_records = records.loc[valid_pressure & valid_age].copy()

# Count exclusions for transparent reporting.
excluded_count = total_records - len(valid_records.index)

# Assign simple teaching categories for readings.
def category_from_row(row):
    if row["systolic"] >= 180 or row["diastolic"] >= 120:
        return "urgent range"
    if row["systolic"] >= 140 or row["diastolic"] >= 90:
        return "high range"
    return "not high range"

# Apply categorization to valid records.
valid_records["category"] = valid_records.apply(category_from_row, axis=1)

# Prepare compact summary statistics.
avg_systolic = valid_records["systolic"].mean()
avg_diastolic = valid_records["diastolic"].mean()

# Count clinically meaningful report categories.
category_counts = valid_records["category"].value_counts().to_dict()

# Build concise report lines.
report_lines = [
    "Blood Pressure Workflow Report",
    f"Records received: {total_records}",
    f"Records analyzed: {len(valid_records.index)}",
    f"Records excluded: {excluded_count}",
    f"Average systolic: {avg_systolic:.1f} mmHg",
    f"Average diastolic: {avg_diastolic:.1f} mmHg",
    f"High range count: {category_counts.get('high range', 0)}",
    f"Urgent range count: {category_counts.get('urgent range', 0)}",
    "Note: report uses deidentified aggregate results only."
]

# Print report without exposing patient codes.
print("\n".join(report_lines))



## **2. Quality Review**

### **2.1. Correctness Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_02_01.jpg?v=1781149616" width="250">



>* Compare outputs with expected medical workflow results
>* Test samples and edge cases for errors

>* Test risky medical data assumptions
>* Make behavior and limits visible

>* Protect sensitive examples and avoid identifiers
>* Report accurate, clear aggregate results



In [ ]:
#@title Python Code - Correctness Checks

# Review checks compare expected and actual outputs.
# Small teaching data avoids sensitive identifiers.
# Aggregate reporting supports safer clinical interpretation.

import pandas as pd

# Create tiny deidentified visit records.
records = [
    {"study_id": "A1", "age": 68, "diagnosis": "Diabetes", "glucose_mgdl": 180},
    {"study_id": "A2", "age": 52, "diagnosis": "diabetes", "glucose_mgdl": 145},

    {"study_id": "A2", "age": 52, "diagnosis": "Diabetes", "glucose_mgdl": 145},
    {"study_id": "A3", "age": None, "diagnosis": "Asthma", "glucose_mgdl": 99},

    {"study_id": "A4", "age": 40, "diagnosis": "Hypertension", "glucose_mgdl": None},
    {"study_id": "A5", "age": 74, "diagnosis": "Diabetes", "glucose_mgdl": 210},
]

# Load records into a small table.
df = pd.DataFrame(records)

# Check required columns before analysis.
required = {"study_id", "age", "diagnosis", "glucose_mgdl"}
missing_columns = required - set(df.columns)

# Stop clearly when required fields are absent.
if missing_columns:
    raise ValueError("Missing required columns for correctness review")

# Remove duplicate patient rows conservatively.
unique_df = df.drop_duplicates(subset=["study_id"])

# Count exclusions with missing required values.
complete_df = unique_df.dropna(subset=["age", "diagnosis", "glucose_mgdl"])
excluded_count = len(unique_df) - len(complete_df)

# Normalize text labels before counting categories.
labels = complete_df["diagnosis"].str.strip().str.lower()
diabetes_count = int((labels == "diabetes").sum())

# Calculate denominator after documented exclusions.
denominator = len(complete_df)
diabetes_percent = diabetes_count / denominator * 100

# Compare results with hand-checked expectations.
expected = {"unique": 5, "complete": 3, "diabetes": 3}
checks = [
    len(unique_df) == expected["unique"],
    denominator == expected["complete"],

    diabetes_count == expected["diabetes"],
]

# Keep output aggregate and concise.
print("Correctness review summary")
print(f"Unique records checked: {len(unique_df)}")
print(f"Excluded incomplete records: {excluded_count}")
print(f"Diabetes among complete records: {diabetes_count}/{denominator}")
print(f"Diabetes percentage: {diabetes_percent:.1f}%")
print(f"All correctness checks passed: {all(checks)}")



### **2.2. Readability Review**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_02_02.jpg?v=1781149618" width="250">



>* Readable code supports safe maintenance
>* Clear structure and names prevent mistakes

>* Show how sensitive data is protected
>* Use comments to explain important choices

>* Use clear flow, names, and explanations
>* Make workflow narrative support safe reuse



In [ ]:
#@title Python Code - Readability Review

# Review readability in a medical workflow.
# Names and sections guide safe maintenance.
# Outputs avoid sensitive record level details.

import pandas as pd

# Create a tiny de-identified teaching dataset.
records = [
    {"age_group": "40-64", "condition": "diabetes", "outcome": "improved"},
    {"age_group": "65+", "condition": "hypertension", "outcome": "stable"},

    {"age_group": "40-64", "condition": "diabetes", "outcome": "stable"},
    {"age_group": "18-39", "condition": "asthma", "outcome": "improved"},
]

# Load records with a meaningful table name.
visit_summary_table = pd.DataFrame(records)

# Define required fields for readability checks.
required_columns = {"age_group", "condition", "outcome"}
missing_columns = required_columns.difference(visit_summary_table.columns)

# Stop early when the workflow is unclear.
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# Summarize only aggregate, de-identified categories.
condition_counts = visit_summary_table["condition"].value_counts().sort_index()
outcome_counts = visit_summary_table["outcome"].value_counts().sort_index()

# Build a short readability review checklist.
readability_checks = {
    "clear purpose stated": True,
    "meaningful names used": True,
    "sensitive fields excluded": True,
    "aggregate report only": True,
}

# Count how many review checks passed.
passed_checks = sum(readability_checks.values())
total_checks = len(readability_checks)

# Print concise review and workflow results.
print("Readability review for de-identified teaching workflow")
print(f"Rows reviewed: {len(visit_summary_table)}")
print(f"Required columns present: {not missing_columns}")
print(f"Checklist passed: {passed_checks} of {total_checks}")
print(f"Conditions summarized: {condition_counts.to_dict()}")
print(f"Outcomes summarized: {outcome_counts.to_dict()}")
print("Interpretation: readable structure supports safer review and reuse.")



### **2.3. Correctness Check**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_02_03.jpg?v=1781149615" width="250">



>* Verify data meaning through every workflow step
>* Check outputs, edge cases, and denominators

>* Test small samples against expected results
>* Verify rules, labels, units, and summaries

>* Catch messy data before reporting results
>* Protect privacy and explain remaining limitations



In [ ]:
#@title Python Code - Correctness Check

# Review correctness using tiny medical examples.
# Expected answers are checked before reporting.
# Sensitive identifiers are excluded from output.

import pandas as pd

# Build a small deidentified visit dataset.
records = [
    {"id": "A001", "age": 64, "systolic": 152, "lab": 8.1},
    {"id": "A002", "age": 51, "systolic": 128, "lab": None},
    {"id": "A003", "age": 200, "systolic": 141, "lab": 6.4},
    {"id": "A001", "age": 64, "systolic": 152, "lab": 8.1},
    {"id": "A004", "age": 45, "systolic": 118, "lab": 5.9},
]

# Convert records into a small table.
df = pd.DataFrame(records)

# Check required columns before calculations.
required = {"id", "age", "systolic", "lab"}
missing_columns = required.difference(df.columns)

# Stop early if structure is unsafe.
if missing_columns:
    raise ValueError("Required columns are missing.")

# Identify records needing review.
bad_age_count = int(((df["age"] < 0) | (df["age"] > 120)).sum())
duplicate_count = int(df.duplicated(subset=["id"]).sum())
missing_lab_count = int(df["lab"].isna().sum())

# Exclude invalid ages and duplicate patients.
clean = df[(df["age"].between(0, 120))].copy()
clean = clean.drop_duplicates(subset=["id"])

# Confirm denominator before summarizing.
denominator = int(len(clean))
if denominator == 0:
    raise ValueError("No valid patients remain.")

# Calculate clinically meaningful counts.
elevated_count = int((clean["systolic"] >= 140).sum())
elevated_percent = round(100 * elevated_count / denominator, 1)
mean_lab = round(clean["lab"].mean(skipna=True), 2)

# Check output against known manual expectations.
expected = {"denominator": 3, "elevated": 1, "missing_lab": 1}
assert denominator == expected["denominator"]
assert elevated_count == expected["elevated"]
assert missing_lab_count == expected["missing_lab"]

# Print a concise quality review report.
print("Correctness check report")
print(f"Valid unique patients counted: {denominator}")
print(f"Elevated systolic count: {elevated_count} ({elevated_percent}%)")
print(f"Mean lab among available values: {mean_lab}")
print(f"Records flagged: age={bad_age_count}, duplicate={duplicate_count}, missing_lab={missing_lab_count}")
print("No direct identifiers or row-level sensitive details printed.")



## **3. Result Presentation**

### **3.1. Summary Findings**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_03_01.jpg?v=1781149604" width="250">



>* Highlight key patterns in plain language
>* Use careful wording to avoid overclaiming

>* Concise findings need context and limits
>* Balance uncertainty with practical meaning

>* Protect privacy by aggregating identifiable details
>* Use respectful language and clear patterns



In [ ]:
#@title Python Code - Summary Findings

# Summaries turn workflow outputs into interpretation.
# Small examples should protect patient privacy.
# Concise findings support responsible medical decisions.

import pandas as pd

# Create a tiny deidentified medical dataset.
records = [
    {"age": 67, "systolic_bp": 148, "diagnosis_group": "cardiometabolic", "followup_done": True},
    {"age": 72, "systolic_bp": 156, "diagnosis_group": "cardiometabolic", "followup_done": False},

    {"age": 54, "systolic_bp": 132, "diagnosis_group": "respiratory", "followup_done": True},
    {"age": 61, "systolic_bp": 144, "diagnosis_group": "cardiometabolic", "followup_done": True},
    {"age": 46, "systolic_bp": 126, "diagnosis_group": "other", "followup_done": False},

    {"age": 79, "systolic_bp": 162, "diagnosis_group": "cardiometabolic", "followup_done": False},
    {"age": 58, "systolic_bp": 138, "diagnosis_group": "respiratory", "followup_done": True},
    {"age": 69, "systolic_bp": 151, "diagnosis_group": "other", "followup_done": True},
]

# Convert records into a dataframe.
df = pd.DataFrame(records)

# Validate required columns before analysis.
needed = {"age", "systolic_bp", "diagnosis_group", "followup_done"}
missing = needed.difference(df.columns)

# Stop early if expected fields are absent.
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Validate the dataset has enough rows.
if len(df) < 3:
    raise ValueError("Dataset is too small for a useful summary.")

# Create clinically meaningful summary variables.
older_percent = (df["age"].ge(65).mean() * 100)
elevated_percent = (df["systolic_bp"].ge(140).mean() * 100)
followup_percent = (df["followup_done"].mean() * 100)

# Summarize follow-up by broad diagnosis group.
group_followup = df.groupby("diagnosis_group")["followup_done"].mean()
lowest_group = group_followup.idxmin()
lowest_rate = group_followup.min() * 100

# Print concise privacy-aware summary findings.
print("Summary findings for a small deidentified sample:")
print(f"Sample size: {len(df)} visits; results are exploratory only.")
print(f"Older adults represented {older_percent:.0f}% of visits.")
print(f"Elevated systolic pressure appeared in {elevated_percent:.0f}% of visits.")
print(f"Overall follow-up completion was {followup_percent:.0f}%.")
print(f"Lowest follow-up was in the {lowest_group} group: {lowest_rate:.0f}%.")
print("Interpretation: describe observed patterns, not causal claims.")
print("Privacy note: categories were broad and no patient details are shown.")



### **3.2. Clinical Interpretation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_03_02.jpg?v=1781149607" width="250">



>* Turn program results into meaningful context
>* Explain patterns without overclaiming conclusions

>* Link findings to cautious real-world meaning
>* Avoid diagnoses; consider context and missingness

>* Protect privacy with careful, aggregated wording
>* Match interpretation to audience and uncertainty



In [ ]:
#@title Python Code - Clinical Interpretation

# Interpret summarized workflow results clinically.
# Use aggregate language for privacy.
# State uncertainty before next steps.

import pandas as pd

# Build a tiny simulated visit dataset.
records = [
    {"age": 45, "systolic": 128, "glucose": 98, "smoking": "No"},
    {"age": 72, "systolic": 154, "glucose": 142, "smoking": "Former"},

    {"age": 67, "systolic": 149, "glucose": 131, "smoking": "No"},
    {"age": 38, "systolic": 118, "glucose": 87, "smoking": None},
]

# Convert records into a small table.
df = pd.DataFrame(records)
visit_count = len(df.index)

# Validate the dataset before summarizing.
required = {"age", "systolic", "glucose", "smoking"}
missing_columns = required.difference(df.columns)

# Stop safely if required fields are absent.
if missing_columns:
    raise ValueError("Required clinical fields are missing")

# Create simple clinical review flags.
older_mask = df["age"] >= 65
elevated_bp = df["systolic"] >= 140

# Summarize results without identifying anyone.
older_mean = df.loc[older_mask, "systolic"].mean()
younger_mean = df.loc[~older_mask, "systolic"].mean()

# Count review flags and missing values.
bp_flags = int(elevated_bp.sum())
glucose_flags = int((df["glucose"] >= 126).sum())
missing_smoking = int(df["smoking"].isna().sum())

# Print concise, clinically cautious interpretation.
print("Clinical interpretation for simulated visits")
print(f"Records summarized: {visit_count}")
print(f"Older average systolic BP: {older_mean:.1f} mmHg")
print(f"Younger average systolic BP: {younger_mean:.1f} mmHg")
print(f"BP review flags: {bp_flags}; glucose review flags: {glucose_flags}")
print(f"Missing smoking status records: {missing_smoking}")
print("Pattern suggests higher BP burden among older visits here.")
print("Flags indicate review needs, not confirmed diagnoses.")
print("Small simulated data limits broad clinical conclusions.")
print("Next step: validate data and review aggregate patterns.")



### **3.3. Summary Findings**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Python for Medicine/Module_05/Lecture_B/image_03_03.jpg?v=1781149606" width="250">



>* Summarize key patterns, exclusions, and sample size
>* Turn raw output into interpretable findings

>* State findings without overstating certainty
>* Protect privacy by reporting aggregate patterns

>* Connect processed data to practical next steps
>* Show data quality and analysis limits



In [ ]:
#@title Python Code - Summary Findings

# Summaries should be concise and transparent.
# Aggregation protects privacy while supporting interpretation.
# Validated records make findings more trustworthy.

import pandas as pd

# Create a tiny simulated encounter dataset.
records = [
    {"id": "A01", "age": 72, "diabetes": True, "systolic": 152},
    {"id": "A02", "age": 45, "diabetes": False, "systolic": 128},
    {"id": "A03", "age": 67, "diabetes": True, "systolic": 146},

    {"id": "A04", "age": None, "diabetes": False, "systolic": 135},
    {"id": "A05", "age": 59, "diabetes": True, "systolic": None},
    {"id": "A06", "age": 81, "diabetes": False, "systolic": 158},
]

# Convert records into a small table.
df = pd.DataFrame(records)

# Define fields needed for this report.
needed = ["age", "diabetes", "systolic"]

# Keep only records with complete report data.
valid = df.dropna(subset=needed).copy()

# Count excluded records for transparency.
excluded_count = len(df) - len(valid)

# Add a clinically interpretable screening flag.
valid["high_systolic"] = valid["systolic"] >= 140

# Summarize only aggregate results.
analyzed_count = len(valid)
older_count = int((valid["age"] >= 60).sum())
diabetes_count = int(valid["diabetes"].sum())
high_bp_count = int(valid["high_systolic"].sum())

# Calculate simple percentages safely.
older_pct = older_count / analyzed_count * 100
high_bp_pct = high_bp_count / analyzed_count * 100

# Present concise findings for interpretation.
print("Summary findings from simulated encounter records")
print(f"Records analyzed: {analyzed_count}; excluded: {excluded_count}.")
print(f"Adults age 60 or older: {older_count} ({older_pct:.0f}%).")
print(f"Documented diabetes: {diabetes_count} of {analyzed_count} records.")
print(f"High systolic flag: {high_bp_count} ({high_bp_pct:.0f}%).")
print("Interpretation: findings describe this small available dataset only.")
print("Privacy note: no individual patient details are reported.")



# <font color="#418FDE" size="6.5" uppercase>**Capstone Case**</font>


In this lecture, you learned to:
- Build a complete Python program that processes a small medical dataset from input to report output. 
- Evaluate the program for correctness, readability, error handling, and appropriate handling of sensitive examples. 
- Present the workflow results in a concise form that supports clinical or research interpretation. 

<font color='yellow'>Congratulations on completing this course!</font>